In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import random

## --- 1. Prepare Dataset and DataLoader ---

# Define data transformations
transform = transforms.ToTensor()

# Download and create the Fashion-MNIST dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Define class names
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')


## --- (Added) Check Training Images Before Start ---

print("## Checking training data samples before starting ##")
# Get the first batch from train_loader
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Create an image grid
# make_grid combines several images into a single grid image
img_grid = torchvision.utils.make_grid(images[:32], nrow=8) # Display 32 images in 8 columns

# Convert tensor to numpy array for visualization
np_img = img_grid.numpy()
plt.figure(figsize=(10, 5))
plt.imshow(np.transpose(np_img, (1, 2, 0)))
plt.title("Fashion-MNIST Training Data Samples")
plt.axis('off')
plt.show()
print("-" * 70)


## --- 2. Define the Neural Network Model ---

class FashionModel(nn.Module):
    def __init__(self):
        super(FashionModel, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = FashionModel()


## --- 3. Define Loss Function and Optimizer ---

learning_rate = 1e-3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


## --- 4. Train the Model ---

print("## Starting Model Training ##")
num_epochs = 5
for epoch in range(num_epochs):
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete!")
print("-" * 70)


## --- 5. Evaluate Model Performance ---

model.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy on the test dataset: {100 * correct / total:.2f} %')
print("-" * 70)


## --- 6. Use the Trained Model (with User Interaction) ---

print("## Predict with the trained model ##")
model.eval() # Set to evaluation mode permanently for inference

while True: # Start an infinite loop
    # Select a random image from the test dataset
    random_idx = random.randint(0, len(test_dataset) - 1)
    test_image, actual_label = test_dataset[random_idx]

    with torch.no_grad():
        # Add a batch dimension to the image for the model
        output = model(test_image.unsqueeze(0))
        predicted_idx = output.argmax(1).item()

    # Visualize the prediction result
    plt.imshow(test_image.squeeze(), cmap="gray")
    plt.title(f"Actual Label: {classes[actual_label]}\nModel Prediction: {classes[predicted_idx]}")
    plt.axis('off')
    plt.show()

    # Ask the user if they want to continue
    user_input = input("\nPress 'y' to test another image, or any other key to exit: ")
    if user_input.lower() != 'y':
        break # Exit the loop if the input is not 'y'

print("\nExiting the program.")